In [3]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [51]:
import pandas as pd
from datetime import datetime
from collections import defaultdict

# Define time period
start = "2025-04-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
type_names = [
    "Address"
]
#, "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=threatAssess,associatedGroups,tags,observations,associatedCases&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

observed_src


Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 8560 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,observations,lastObserved,privateFlag,active,activeLocked,ip,legacyLink,tags.data,associatedGroups.data,indicator
0,234703,2017-01-03T15:27:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-05T13:57:40Z,5.0,98.0,2.68,...,2608,2025-05-04T00:00:00Z,False,True,False,185.65.134.76,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 37388, 'name': 'Midnight Blizzard', 'l...","[{'id': 157678, 'dateAdded': '2023-08-02T13:24...",185.65.134.76
1,4515496,2024-02-01T17:29:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-05T13:57:37Z,3.0,44.0,3.00,...,473062,2025-05-04T00:00:00Z,False,True,False,68.71.247.130,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 240980, 'name': 'Alert ID : ba5f4916',...","[{'id': 312909, 'dateAdded': '2024-02-01T17:29...",68.71.247.130
2,5265103,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-05T13:57:36Z,3.0,70.0,3.00,...,538834,2025-05-04T00:00:00Z,False,True,False,165.232.138.158,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 546888, 'dateAdded': '2025-01-23T19:51...",165.232.138.158
3,234726,2017-01-03T15:27:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-05T13:57:36Z,5.0,98.0,2.19,...,64216,2025-05-05T00:00:00Z,False,True,False,185.129.62.63,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 37388, 'name': 'Midnight Blizzard', 'l...","[{'id': 157678, 'dateAdded': '2023-08-02T13:24...",185.129.62.63
4,6755399443033273,2025-04-11T14:29:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-05T13:57:35Z,4.0,80.0,4.00,...,97856,2025-05-05T00:00:00Z,False,True,False,165.232.153.27,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 473166, 'name': 'Malware: GammaSteel',...","[{'id': 311499, 'dateAdded': '2024-01-26T18:09...",165.232.153.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,4252463,2022-09-13T12:57:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2024-01-17T09:49:46Z,3.0,0.0,2.67,...,0,NaN,False,True,False,64.188.27.73,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33387, 'name': 'VMWare Horizon', 'last...","[{'id': 141317, 'dateAdded': '2022-09-13T12:05...",64.188.27.73
9996,4252462,2022-09-13T12:57:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2024-01-17T09:49:45Z,3.0,0.0,2.67,...,0,NaN,False,True,False,52.202.193.124,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33387, 'name': 'VMWare Horizon', 'last...","[{'id': 141317, 'dateAdded': '2022-09-13T12:05...",52.202.193.124
9997,4245414,2022-08-09T13:42:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2024-01-17T09:49:43Z,3.0,0.0,1.50,...,1549,2023-01-03T00:00:00Z,False,True,False,39.104.90.45,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33418, 'name': 'exploitation tools', '...","[{'id': 140048, 'dateAdded': '2022-08-09T13:38...",39.104.90.45
9998,4496700,2023-12-31T08:55:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2024-01-14T04:35:50Z,NaN,NaN,0.00,...,0,NaN,False,True,False,44.215.45.80,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,44.215.45.80


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    if 'tags.data' in row and isinstance(row['tags.data'], list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(row['tags.data'])
        
        # Add 'summary', 'observations', and 'description' columns to the tags
        tags['summary'] = row['summary']
        tags['observations'] = row.get('observations', None)
        tags['description'] = row.get('description', None)
        tags['type'] = row.get('type', None)
        
        # Filter tags containing 'API' (case-insensitive)
        filtered = tags[tags['name'].str.contains('API', case=False, na=False)]
        
        # Append the filtered tags to the result DataFrame
        filtered_tags = pd.concat([filtered_tags, filtered], ignore_index=True)
        
# Ensure datetime format
filtered_tags['lastUsed'] = pd.to_datetime(filtered_tags['lastUsed'])

# Define cutoff time in UTC (timezone-aware)
cutoff = pd.Timestamp.utcnow()

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastUsed'] >= cutoff - pd.Timedelta(hours=48)].copy()


# Extract partner names
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Get the most recent lastUsed per indicator
latest_dates = (
    recent_tags.groupby('summary')['lastUsed']
    .max()
    .reset_index()
    .rename(columns={'lastUsed': 'mostRecentLastUsed'})
)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge everything
result = partner_groups.merge(latest_dates, on='summary', how='left')

# Filter to partner_count >= 2
result = result[result['partner_count'] >= 2]

# Merge optional description/observation info
meta_cols = recent_tags.drop_duplicates('summary')[['summary', 'description', 'observations', 'type']]
result = result.merge(meta_cols, on='summary', how='left')
result = result[result['observations'] < 10000]

# Sort by most recent use
result = result.sort_values(by='mostRecentLastUsed', ascending=False)


In [78]:
specific_indicator = result[result['summary'] == '1.233.75.195']
specific_indicator

,summary,partner_count,partners,mostRecentLastUsed,description,observations,type
0,1.233.75.195,3,"CMS, HRSA, OS",2025-05-05 13:51:45+00:00,Health-ISAC is sharing these IOCs to increase ...,6114,Address


In [93]:
import os
import json
import time
import urllib.parse
import urllib3
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

#VT_API_KEY - a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08
#OTX_API_KEY - ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# === CONFIG ===
FILE_PATH = r'C:\Users\jaskew\Documents\project_repository\data\processed\BingSearchData.json'
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"


# === Helper Functions ===

def load_saved_links(file_path):
    if not os.path.exists(file_path):
        return set()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return {entry['link'] for entry in data if 'link' in entry}
    except (json.JSONDecodeError, IOError):
        return set()

def fetch_virustotal_data(ip):
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error querying VirusTotal API for {ip}: {e}")
        return None

def fetch_otx_data(ip):
    url = f"https://otx.alienvault.io/api/v1/indicators/IPv4/{ip}/general"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    
    try:
        # First try with the correct domain
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.SSLError as ssl_err:
        print(f"SSL Error occurred: {ssl_err}")
        # If SSL verification fails, we can try with verify=False (for testing only)
        print("Attempting request with SSL verification disabled (not recommended for production)...")
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying OTX API for {ip}: {e}")
        return None

def save_to_json(entry, file_path):
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                except json.JSONDecodeError:
                    data = []
        else:
            data = []

        data.append(entry)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"Error saving data: {e}")

# === Main Script ===

def main():
    saved_links = load_saved_links(FILE_PATH)
    print(f"Loaded {len(saved_links)} previously saved links.")

    search_terms = ['65.21.203.39']
    
    for indicator in search_terms:
        print(f"\n Searching for: {indicator}")
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        # === VIRUSTOTAL ===
        vt_url = f"https://www.virustotal.com/gui/ip-address/{indicator}"
        vt_data = fetch_virustotal_data(indicator)

        if vt_data:
            attributes = vt_data.get("data", {}).get("attributes", {})
            services = attributes.get("services", [])
            ports = [s.get("port") for s in services if "port" in s]
            vt_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"VirusTotal API Report for {indicator}",
                'link': vt_url,
                'asn': attributes.get('asn'),
                'as_owner': attributes.get('as_owner'),
                'country': attributes.get('country'),
                'network': attributes.get('network'),
                'last_analysis_stats': attributes.get('last_analysis_stats'),
                'reputation': attributes.get('reputation'),
                'total_votes': attributes.get('total_votes'),
                'open_ports': ports,
                'publish_date': "N/A"
            }

            if vt_url not in saved_links:
                save_to_json(vt_entry, FILE_PATH)
                saved_links.add(vt_url)
                print(f"Saved VT report for {indicator}.")
            else:
                print(f"VirusTotal entry for {indicator} already saved.")

        # === OTX ===
        otx_url = f"https://otx.alienvault.com/api/v1/indicators/IPv4/{indicator}/general"
        otx_data = fetch_otx_data(indicator)
    
        if otx_data:
            otx_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'title': f"OTX API Report for {indicator}",
                'link': otx_url,
                'base_indicator': otx_data.get('base_indicator', {}),
                'reputation': otx_data.get('reputation'),
                'pulse_info': otx_data.get('pulse_info', {}),
                'geo': otx_data.get('geo', {}),
                'whois': otx_data.get('whois', {}),
                'open_ports': otx_data.get('open_ports', []),
                'publish_date': "N/A"
            }

            if otx_url not in saved_links:
                save_to_json(otx_entry, FILE_PATH)
                saved_links.add(otx_url)
                print(f"Saved OTX report for {indicator}.")
            else:
                print(f"OTX entry for {indicator} already saved.")

if __name__ == "__main__":
    main()


[nltk_data] Error loading stopwords: HTTP Error 503: Service
[nltk_data]     Unavailable


Loaded 0 previously saved links.

 Searching for: 65.21.203.39
Saved VT report for 65.21.203.39.
SSL Error occurred: HTTPSConnectionPool(host='otx.alienvault.io', port=443): Max retries exceeded with url: /api/v1/indicators/IPv4/65.21.203.39/general (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1002)')))
Attempting request with SSL verification disabled (not recommended for production)...
Saved OTX report for 65.21.203.39.


In [ ]:
from docx import Document

def fill_word_template(template_path, output_path, replacements):
    doc = Document(template_path)

    # Replace placeholders in paragraphs
    for para in doc.paragraphs:
        for key, value in replacements.items():
            if key in para.text:
                para.text = para.text.replace(key, value)

    # Replace placeholders in tables (if needed)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for key, value in replacements.items():
                    if key in cell.text:
                        cell.text = cell.text.replace(key, value)

    doc.save(output_path)

# Example usage
fill_word_template(
    template_path=r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx",
    output_path=r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\output.docx"
,
    replacements={
        "{{title}}": "I&W Report Generated",
        "{{ip_address}}": "65.21.203.39",
        "{{indicator_types}}": "IPv4 Address",
        "{{date_observed}}": "08/01/2024",
        "{{observed_by}}": "FDA, HHS, CMS, HRSA, OS, VA",
        "{{sources}}": 'www.testResource.com',
        "{{asn_number}}": '2923',
        "{{as_owner}}": 'Hetzner Online GmbH'

    }
)
